In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_excel('Data_Mahasiswa_UNMUH.xlsx')

df.head()

,Nama,Jenis Kelamin,Program Studi,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,...,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi,Estimasi Masa Studi
0,Sinta,Perempuan,PGSD,2020,2024,3.70,3.62,3.66,3.75,3.59,...,NaN,151,0,rendah,sedang,sedang,menengah,bekerja,tidak aktif,4.6
1,Sasi Mardita,Perempuan,PGSD,2021,2025,3.78,3.62,3.65,3.84,3.90,...,NaN,151,0,sangat tinggi,sangat tinggi,rendah,menengah,tidak bekerja,aktif,4.0
2,Muhammad Farhan Sugiwangsa,Laki-laki,Teknik Sipil,2021,2025,3.60,3.60,3.65,3.70,3.68,...,NaN,144,0,sedang,sedang,sedang,menengah,tidak bekerja,tidak aktif,3.7
3,Lilis Karsina,Perempuan,KSDA,2020,2025,3.48,3.30,3.40,3.35,3.45,...,NaN,145,0,rendah,sedang,sedang,menengah,tidak bekerja,tidak aktif,4.3
4,Muhammad Miraldy,Laki-laki,Teknik Sipil,2021,2025,3.39,3.25,3.30,3.30,3.35,...,NaN,146,0,rendah,tinggi,sedang,tinggi,tidak bekerja,tidak aktif,3.8


In [ ]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Nama                             278 non-null    object 
 1   Jenis Kelamin                    278 non-null    object 
 2   Program Studi                    278 non-null    object 
 3   Tahun Masuk                      278 non-null    int64  
 4   Tahun Lulus                      278 non-null    int64  
 5   IPK                              278 non-null    float64
 6   IPS 1                            278 non-null    float64
 7   IPS 2                            278 non-null    float64
 8   IPS 3                            278 non-null    float64
 9   IPS 4                            278 non-null    float64
 10  IPS 5                            278 non-null    float64
 11  IPS 6                            278 non-null    float64
 12  IPS 7                 

,0
Nama,0
Jenis Kelamin,0
Program Studi,0
Tahun Masuk,0
Tahun Lulus,0
IPK,0
IPS 1,0
IPS 2,0
IPS 3,0
IPS 4,0


In [ ]:
# mapping manual
mapping_5 = {
    'sangat rendah':1,'rendah':2,'sedang':3,'tinggi':4,'sangat tinggi':5
}

mapping_stres = {'rendah':1,'sedang':2,'tinggi':3}
mapping_ekonomi = {'rendah':1,'menengah':2,'tinggi':3}
mapping_kerja = {'tidak bekerja':0,'bekerja':1}
mapping_org = {'tidak aktif':0,'aktif':1}

# kolom kategori
cols_kategori = [
    'Motivasi Belajar',
    'Dukungan Keluarga',
    'Tingkat Stres',
    'Sosial-Ekonomi',
    'Pekerjaan Paruh Waktu',
    'Keaktifan dalam Berorganisasi'
]

# ubah ke huruf kecil
for col in cols_kategori:
    df[col] = df[col].astype(str).str.lower().str.strip()

# mapping
df['Motivasi Belajar'] = df['Motivasi Belajar'].map(mapping_5)
df['Dukungan Keluarga'] = df['Dukungan Keluarga'].map(mapping_5)
df['Tingkat Stres'] = df['Tingkat Stres'].map(mapping_stres)
df['Sosial-Ekonomi'] = df['Sosial-Ekonomi'].map(mapping_ekonomi)
df['Pekerjaan Paruh Waktu'] = df['Pekerjaan Paruh Waktu'].map(mapping_kerja)
df['Keaktifan dalam Berorganisasi'] = df['Keaktifan dalam Berorganisasi'].map(mapping_org)


In [ ]:
ips_cols = [f'IPS {i}' for i in range(1, 12)]

# pastikan semua IPS ada
for col in ips_cols:
    if col not in df.columns:
        df[col] = np.nan

# isi yang kosong dengan rata-rata
df[ips_cols] = df[ips_cols].apply(
    lambda row: row.fillna(row.mean()), axis=1
)


In [ ]:
df['IPK'] = df['IPK'].clip(0, 4)
df['Jumlah Mata Kuliah yang Diulang'] = df['Jumlah Mata Kuliah yang Diulang'].clip(0, 10)

In [ ]:
df = df.drop([
    'Nama',
    'Program Studi',
    'Jenis Kelamin',
], axis=1, errors='ignore')

In [ ]:
y = df['Estimasi Masa Studi']
X = df.drop('Estimasi Masa Studi', axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train

,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,IPS 5,IPS 6,IPS 7,...,IPS 10,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi
260,2021,2025,3.79,3.75,3.78,3.80,3.82,3.79,3.81,3.77,...,3.78875,3.78875,144,0,2,3,1,3.0,0,0
124,2020,2024,3.60,3.45,3.50,3.55,3.60,3.68,3.65,3.70,...,3.59500,3.59500,148,0,5,5,3,3.0,0,0
33,2021,2025,3.91,3.95,3.90,3.80,3.85,3.92,3.98,3.94,...,3.91000,3.91000,140,0,4,4,1,NaN,0,1
86,2020,2024,3.81,3.68,3.63,3.71,3.83,3.88,3.86,3.84,...,3.79125,3.79125,151,0,3,4,2,2.0,1,0
264,2021,2025,3.49,3.40,3.45,3.48,3.50,3.47,3.51,3.49,...,3.47750,3.47750,145,0,3,3,2,2.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,2021,2025,3.80,3.69,3.75,3.88,3.82,3.85,3.82,3.79,...,3.80375,3.80375,151,0,3,3,1,3.0,0,0
71,2020,2024,3.82,3.60,3.70,3.78,3.84,3.90,3.95,3.91,...,3.81875,3.81875,151,0,4,4,2,2.0,0,0
106,2020,2024,3.79,3.62,3.71,3.76,3.80,3.85,3.89,3.87,...,3.79625,3.79625,151,0,3,3,2,2.0,0,0
270,2021,2025,3.81,3.77,3.79,3.81,3.82,3.84,3.80,3.86,...,3.81000,3.81000,146,0,3,3,2,2.0,0,0


In [ ]:
X_test

,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,IPS 5,IPS 6,IPS 7,...,IPS 10,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi
30,2020,2024,3.40,3.30,3.30,3.20,3.35,3.40,3.40,3.55,...,3.38125,3.38125,152,0,3,3,2,1.0,0,1
126,2021,2025,3.93,3.90,3.92,3.95,4.00,3.98,3.94,3.90,...,3.93000,3.93000,146,0,5,5,2,3.0,0,0
199,2021,2025,3.79,3.72,3.78,3.85,3.88,3.83,3.80,3.77,...,3.79625,3.79625,151,0,3,3,1,3.0,0,0
142,2021,2025,3.83,3.75,3.80,3.86,3.88,3.85,3.83,3.81,...,3.83500,3.83500,151,0,3,4,1,2.0,0,1
253,2021,2025,3.84,3.82,3.80,3.88,3.85,3.83,3.84,3.86,...,3.84000,3.84000,146,0,3,3,1,3.0,0,0
237,2020,2025,3.79,3.65,3.72,3.76,3.83,3.80,3.88,3.93,...,3.75000,3.79000,151,0,2,3,1,3.0,0,0
97,2020,2024,3.76,3.66,3.71,3.76,3.81,3.85,3.83,3.79,...,3.76000,3.76000,151,0,2,1,1,2.0,0,0
206,2021,2025,3.87,3.78,3.84,3.89,3.93,3.96,3.91,3.88,...,3.87000,3.87000,151,0,5,4,1,2.0,0,0
263,2021,2025,3.61,3.55,3.58,3.60,3.63,3.59,3.62,3.61,...,3.60250,3.60250,144,0,3,3,2,2.0,0,1
144,2021,2025,3.83,3.76,3.82,3.88,4.00,3.78,3.84,3.80,...,3.83875,3.83875,151,0,3,4,1,2.0,0,0


In [ ]:
model = RandomForestRegressor(
    n_estimators=700,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=700, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nEvaluasi Model:")
print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)


Evaluasi Model:
MAE: 0.15399723492487258
MSE: 0.0382226857026229
R2 Score: 0.743519113376644


In [ ]:
joblib.dump(model, 'model_random_forest.pkl')

['model_random_forest.pkl']

In [ ]:
joblib.dump(X.columns.tolist(), "fitur_model.pkl")

print("\n✅ Model & fitur berhasil disimpan!")


✅ Model & fitur berhasil disimpan!


In [ ]:
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the trained model
loaded_model = joblib.load('model_random_forest.pkl')

# Load the feature names used during training
loaded_features = joblib.load('fitur_model.pkl')

# Ensure X_test columns match the loaded_features order
X_test_aligned = X_test[loaded_features]
# Make predictions using the loaded model
loaded_model_predictions = loaded_model.predict(X_test_aligned)

# Display some predictions
print("\n--- Hasil Prediksi dari Model yang Dimuat ---")
print("Prediksi: ", loaded_model_predictions[:5])
print("Aktual: ", y_test.head().values)

# Evaluate the loaded model
mae_loaded = mean_absolute_error(y_test, loaded_model_predictions)
mse_loaded = mean_squared_error(y_test, loaded_model_predictions)
r2_loaded = r2_score(y_test, loaded_model_predictions)

print("\n--- Evaluasi Model yang Dimuat ---")
print(f"MAE (Loaded Model): {mae_loaded}")
print(f"MSE (Loaded Model): {mse_loaded}")
print(f"R2 Score (Loaded Model): {r2_loaded}")


--- Hasil Prediksi dari Model yang Dimuat ---
Prediksi:  [4.19484659 3.69928571 4.15362046 3.94060865 3.98793414]
Aktual:  [4.1 3.9 4.2 4.  4.4]

--- Evaluasi Model yang Dimuat ---
MAE (Loaded Model): 0.15399723492487258
MSE (Loaded Model): 0.0382226857026229
R2 Score (Loaded Model): 0.743519113376644
